# Analiza portfela XTB

Wczytaj eksporty CSV do `data/`, uruchom wszystkie komórki. Wykresy pokazują:
- wartość portfela z widocznymi skokami wpłat,
- porównanie zwrotu z S&P 500 TR i NASDAQ 100 TR (te same wpłaty w te same dni),
- koszt przewalutowań (prowizja XTB 0,5%),
- alokację sektorową (wykres kołowy),
- aktualną stopę zwrotu.

> **Setup:** zalecane `pip install -e ".[dev]"` z katalogu głównego projektu i restart kernela. Notebook dodaje też `src/` do ścieżki importu.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from IPython.display import display

from xtb_analyzer.benchmark import build_deposit_benchmarks
from xtb_analyzer.charts import (
    PLOTLY_SHOW_CONFIG,
    prepare_return_comparison_data,
    show_return_comparison_widget,
    plot_conversion_losses,
    plot_nav_with_deposits,
    plot_return_dashboard,
    plot_sector_allocation,
)
from xtb_analyzer.flows import analyze_conversion_losses, attach_deposits_to_nav, conversion_summary
from xtb_analyzer.metrics import (
    build_report,
    compute_metrics,
    format_metrics_table,
    format_realized_pnl_table,
    realized_pnl_summary,
)
from xtb_analyzer.parser import load_accounts
from xtb_analyzer.portfolio import build_nav_series
from xtb_analyzer.sectors import enrich_positions_with_sectors, sector_allocation

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
REPORT_CURRENCY = "pln"
REFRESH_MARKET_DATA = True

CSV_FILES = sorted(DATA_DIR.glob("XTB_*_account_cs.csv"))
if not CSV_FILES:
    raise FileNotFoundError("Brak plikow CSV w data/.")

accounts = load_accounts(CSV_FILES)
nav, holdings_ts, positions = build_nav_series(accounts, live_prices=REFRESH_MARKET_DATA)
nav = attach_deposits_to_nav(nav, accounts)
positions = enrich_positions_with_sectors(positions)
conversion_losses = analyze_conversion_losses(accounts)
conversion_stats = conversion_summary(conversion_losses)
sp500_benchmark, nasdaq100_benchmark = build_deposit_benchmarks(
    accounts,
    nav.index,
    currency=REPORT_CURRENCY,
    refresh_latest=REFRESH_MARKET_DATA,
)
return_comparison_data = prepare_return_comparison_data(
    nav, sp500_benchmark, nasdaq100_benchmark, REPORT_CURRENCY
)

print("Konta:", [a.account_id for a in accounts])
print("Okres:", nav.index[0].date(), "->", nav.index[-1].date())


Konta: ['55287715', '53287226']
Okres: 2025-11-04 -> 2026-07-05


## Aktualna stopa zwrotu

In [2]:
plot_return_dashboard(nav, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)

## Wartość portfela w czasie (ze skokami wpłat)

In [3]:
plot_nav_with_deposits(nav, REPORT_CURRENCY).show(config=PLOTLY_SHOW_CONFIG)

## Zwrot z kapitału vs benchmarki (dane miesięczne)


In [4]:
show_return_comparison_widget(
    nav,
    sp500_benchmark,
    nasdaq100_benchmark,
    REPORT_CURRENCY,
    data=return_comparison_data,
)


## Koszt przewalutowań (prowizja XTB 0,5%)

In [5]:
plot_conversion_losses(conversion_losses, conversion_stats).show(config=PLOTLY_SHOW_CONFIG)

## Pozycje, sektory i metryki ryzyka

In [6]:
plot_sector_allocation(sector_allocation(positions)).show(config=PLOTLY_SHOW_CONFIG)

display(positions.sort_values("market_value_pln", ascending=False))
display(format_metrics_table(compute_metrics(nav, REPORT_CURRENCY), currency=REPORT_CURRENCY))
display(format_realized_pnl_table(realized_pnl_summary(accounts)))

,account_id,symbol,currency,shares,avg_cost,market_price,cost_basis,market_value,unrealized_pnl,market_value_pln,market_value_usd,market_value_eur,weight,sector,industry
4,53287226,NVDA.US,USD,5.6500,210.704071,194.830002,1190.478000,1100.789510,-89.688490,4125.428821,1100.789510,962.200173,0.294029,Technology,Semiconductors
0,53287226,UBER.US,USD,13.0000,84.765286,74.430000,1101.948712,967.590004,-134.358708,3626.237035,967.590004,845.770477,0.258451,Technology,Software - Application
2,53287226,NOW.US,USD,5.0000,103.920124,106.320000,519.600622,531.599998,11.999376,1992.277302,531.599998,464.671589,0.141994,Technology,Software - Application
1,53287226,GOOGL.US,USD,1.0000,366.640445,359.910004,366.640445,359.910004,-6.730441,1348.834712,359.910004,314.597354,0.096135,Communication Services,Internet Content & Information
8,53287226,CRDO.US,USD,1.0000,221.920000,241.910004,221.920000,241.910004,19.990004,906.606115,241.910004,211.453548,0.064616,Technology,Semiconductors
3,55287715,IFX.DE,EUR,2.2546,78.058871,77.379997,175.991530,174.460942,-1.530588,747.914089,199.589212,174.460942,0.053306,Technology,Semiconductors
5,53287226,NU.US,USD,12.6000,12.923492,13.610000,162.836000,171.485996,8.649996,642.678062,171.485996,149.895918,0.045805,Financial Services,Banks - Regional
7,53287226,ACMR.US,USD,1.3006,83.002357,97.769997,107.952866,127.159658,19.206792,476.556246,127.159658,111.150264,0.033965,Technology,Semiconductor Equipment & Materials
6,53287226,POET.US,USD,5.0000,12.790000,8.760000,63.950000,43.800001,-20.149999,164.149263,43.800001,38.285583,0.011699,Technology,Semiconductors


,Metryka,Wartość
0,Aktualna wartość portfela,"14 036,63 PLN"
1,Suma wpłat,"10 839,69 PLN"
2,Zysk / strata,"3 196,93 PLN"
3,Zwrot na wpłaconym kapitale,"+29,49% (wartość ÷ wpłaty − 1)"
4,Zmienność (roczna),"+1047,13%"
5,Współczynnik Sharpe'a,"2,78"
6,Współczynnik Sortino,"10,88"
7,Maks. obsunięcie (drawdown),"-20,36%"
8,Szczyt wartości portfela (ATH),"14 238,09 PLN"
9,Odległość od ATH,"-1,41%"


,Symbol,Transakcje,Zrealizowany P&L,Średni P&L,Win rate
0,AMD.US,10,1012.08,101.21,100.0
1,AMZN.US,4,-2.99,-0.75,25.0


## Eksport

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
build_report(accounts, nav, positions, output_dir=OUTPUT_DIR)
nav.to_csv(OUTPUT_DIR / "nav_timeseries.csv")
conversion_losses.to_csv(OUTPUT_DIR / "conversion_losses.csv", index=False)
sp500_benchmark.to_csv(OUTPUT_DIR / "sp500_benchmark.csv")
nasdaq100_benchmark.to_csv(OUTPUT_DIR / "nasdaq100_benchmark.csv")
print(f"Zapisano do {OUTPUT_DIR.resolve()}")

Zapisano do C:\Users\oliwi\OneDrive\Desktop\xtb_portfolio_analyzer\output
